# Install packages

In [ ]:

import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from functools import partial
import torch
from tqdm.auto import tqdm

# Random Seed

In [ ]:
from torch import manual_seed as torch_manual_seed
import random
import numpy as np

from torch.cuda import manual_seed_all
from torch.backends import cudnn

def setup_seed(seed):
    torch_manual_seed(seed)
    manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    cudnn.deterministic = True

SEED = 42
setup_seed(SEED)

# Loading Images

In [16]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class YoloToEfficientNetDataset(Dataset):
    def __init__(self, images_dir, labels_dir, transform=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transform = transform
        
        # Grab all .jpg files
        self.image_files = [f for f in os.listdir(images_dir) if f.endswith('.jpg')]
        
        # Define the class names based on your YAML
        self.classes = ['boat', 'human', 'other', 'shark']

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        # 1. Image loading
        img_name = self.image_files[idx]
        img_path = os.path.join(self.images_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        
        # 2. Label parsing (Handling YOLO format)
        label_name = img_name.replace('.jpg', '.txt')
        label_path = os.path.join(self.labels_dir, label_name)
        
        class_id = -1 # Default/fallback
        
        # Read the YOLO txt file
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                line = f.readline().strip()
                if line:
                    # YOLO format: <class_id> <x> <y> <w> <h>
                    # We split by space and grab the very first item
                    class_id = int(line.split()[0])
        
        # Convert to tensor. If no label is found, you might want to handle it differently 
        # (e.g., assigning to a 'background' class or dropping the image entirely).
        label = torch.tensor(class_id, dtype=torch.long)

        # 3. Apply standard EfficientNet transforms
        if self.transform:
            image = self.transform(image)

        return image, label

# --- Setup and Testing ---

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

images_directory = r"data\raw\train\images"
labels_directory = r"data\raw\train\labels"

train_dataset = YoloToEfficientNetDataset(
    images_dir=images_directory, 
    labels_dir=labels_directory, 
    transform=train_transforms
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_images_directory = r"data\raw\valid\images"
val_labels_directory = r"data\raw\valid\labels"

test_images_directory = r"data\raw\test\images"
test_labels_directory = r"data\raw\test\labels"


val_dataset = YoloToEfficientNetDataset(
    images_dir=val_images_directory, 
    labels_dir=val_labels_directory, 
    transform=train_transforms
)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

test_dataset = YoloToEfficientNetDataset(
    images_dir=test_images_directory, 
    labels_dir=test_labels_directory, 
    transform=train_transforms
)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Tansfer Learning Packages

In [17]:
from torch.nn import Module
import torch.optim as optim
import torch.nn as nn
import timm

# Efficient Net B0 CNN

In [18]:
class EffNetB0CNN(Module):
    def __init__(self, num_classes = 4):
        super(EffNetB0CNN, self).__init__()
        self.model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=num_classes)

    def forward(self, X):
      output = self.model(X)
      return output


## Training

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
num_epochs = 5
model = EffNetB0CNN(num_classes=4).to(device)

print(f"Using device:{device}")


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_losses, val_losses = [], []
train_accs, val_accs = [], []

best_val_acc = 0

for epoch in range(num_epochs):
  model.train()
  running_loss, correct_train, total_train =0, 0, 0

  train_loader_tqdm = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")

  for images, labels in train_loader_tqdm:
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad()
    predictions = model(images)
    loss = criterion(predictions, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
    _, predicted = torch.max(predictions.data, 1)
    total_train += labels.size(0)
    correct_train += (predicted == labels).sum().item()

    train_loader_tqdm.set_postfix(loss=loss.item())

  model.eval()
  val_loss, correct_val, total_val = 0, 0, 0

  with torch.no_grad():
    val_loader_tqdm = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")
    for images, labels in val_loader_tqdm:
      images, labels = images.to(device), labels.to(device)
      outputs = model(images)
      loss = criterion(outputs, labels)

      val_loss += loss.item()
      _, predicted = torch.max(outputs.data, 1)
      total_val += labels.size(0)
      correct_val += (predicted == labels).sum().item()


  epoch_train_loss = running_loss / len(train_loader)
  epoch_val_loss = val_loss / len(val_loader)
  epoch_train_acc = 100 * correct_train / total_train
  epoch_val_acc = 100 * correct_val / total_val

  if epoch_val_acc > best_val_acc:
    best_val_acc = epoch_val_acc
    torch.save(model.state_dict(), 'best_model.pth')

  train_losses.append(epoch_train_loss)
  val_losses.append(epoch_val_loss)
  train_accs.append(epoch_train_acc)
  val_accs.append(epoch_val_acc)

  print(f"\n Summary of Epoch {epoch+1}:")
  print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}%")
  print(f"Validation Loss: {epoch_val_loss:.4f} | Validation Acc: {epoch_val_acc:.2f}%")
#Final Test after Training
model.load_state_dict(torch.load('best_model.pth'))
model.eval()
final_test_correct = 0
final_test_total = 0

with torch.no_grad():
  for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    _, predicted = torch.max(outputs.data, 1)
    final_test_total += labels.size(0)
    final_test_correct += (predicted == labels).sum().item()

final_accuracy = 100 * final_test_correct / final_test_total
print(f"\n Final Evaluation on Test Set: {final_accuracy:.2f}")

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Using device:cpu


c:\Users\mason\Spring2026\DS6050_ML3_Final_Proj\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mason\.cache\huggingface\hub\models--timm--efficientnet_b0.ra_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Epoch 1/10 [Train]:   0%|          | 0/181 [00:00<?, ?it/s]

KeyboardInterrupt: 